# QSense NN Training on Kaggle GPU

**Instructions:**
1. Upload the full QSense repo as a Kaggle Dataset or use the GitHub integration.
2. Select a GPU accelerator (T4 or P100) in the notebook settings.
3. Run all cells.
4. Download `artifacts/nn_weights.npz` and `artifacts/nn_meta.json` from the output.

Expected training time: ~10-15 min on T4 GPU for 20k steps.

In [ ]:
# If running from Kaggle with the repo as a dataset, adjust the path below
import sys, os

# Detect Kaggle environment
if os.path.exists('/kaggle/input'):
    # Find the repo directory in Kaggle input
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train_nn.py' in files or 'kaggle_train.py' in files:
            # Navigate up to repo root if needed
            if 'train' in dirs or any(f.endswith('.py') for f in files):
                repo_root = root
                break
    else:
        # Fallback: copy files
        repo_root = '/kaggle/working'
    print(f'Repo root: {repo_root}')
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    os.chdir(repo_root)
else:
    # Local run
    repo_root = os.getcwd()
    print(f'Local run from: {repo_root}')

print('Python path:', sys.path[:3])

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'],
                       capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else 'torch already installed')

In [ ]:
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from pathlib import Path

# Create artifacts directory in working directory
artifacts_dir = Path('/kaggle/working/artifacts') if os.path.exists('/kaggle') else Path('artifacts')
artifacts_dir.mkdir(parents=True, exist_ok=True)
print(f'Artifacts will be saved to: {artifacts_dir}')

In [ ]:
from train.train_nn import train

model, best_val_nll = train(
    steps      = 20_000,
    batch      = 2048,
    lr         = 1e-3,
    lr_min     = 1e-5,
    val_every  = 500,
    out        = artifacts_dir / 'nn_weights.npz',
    seed       = 20261004,
    device_str = 'auto',
)
print(f'\nTraining complete. Best validation NLL: {best_val_nll:.4f}')

In [ ]:
# Quick sanity check: numpy inference
import numpy as np
from est.nn_infer import QSenseNetNumpy, _bin_centers

np_model = QSenseNetNumpy(artifacts_dir / 'nn_weights.npz')

# Random test input
rng = np.random.default_rng(42)
x = rng.uniform(0, 1, (10, 19)).astype(np.float32)
probs = np_model.forward(x)
print(f'Output shape: {probs.shape}')
print(f'Sum check: {probs.sum(axis=1).round(6)}')
omega_ests = (probs * _bin_centers[None, :]).sum(axis=1)
print(f'Omega estimates (rad/us): {omega_ests.round(4)}')

In [ ]:
# Torch vs numpy equivalence check
import torch
import torch.nn.functional as F
from train.train_nn import QSenseNet

state = {k: torch.from_numpy(v) for k, v in np.load(str(artifacts_dir / 'nn_weights.npz')).items()}
torch_model = QSenseNet()
torch_model.load_state_dict(state)
torch_model.eval()

x_th = torch.from_numpy(x)
with torch.no_grad():
    torch_probs = F.softmax(torch_model(x_th), dim=-1).numpy()

max_diff = np.abs(torch_probs - probs).max()
print(f'Max |torch - numpy| = {max_diff:.2e}')
assert max_diff < 1e-5, f'FAILED: difference {max_diff:.2e} > 1e-5'
print('✓ Numpy forward pass matches torch within 1e-5')

In [ ]:
# List output files
for f in sorted(artifacts_dir.glob('*')):
    print(f'{f.name:30} {f.stat().st_size / 1024:.1f} KB')